# Full Fine-Tuning Baseline ResNet-50 with 5-Fold CV

This notebook trains a baseline ResNet-50 without segmentation guidance. The complete ImageNet-pretrained backbone and classifier are fine-tuned. It uses the same CT samples, five patient-isolated folds, holdout set, preprocessing, optimizer settings, and model-selection criterion as the multi-stage direct-guided experiment.

After training, the five best fold checkpoints are ensembled on the holdout set. Grad-CAM and LRP are generated for layer1, layer2, layer3, and layer4. Progress is displayed with tqdm.

## Archive required in Google Drive

Place these two files directly under MyDrive/mask-guided-lung-nodule-xai/:

fa028196-fd4c-441f-a5d3-3efb9707e7f9_guided_resnet50_parenchyma.tar.gz

fa028196-fd4c-441f-a5d3-3efb9707e7f9_guided_resnet50_parenchyma.tar.gz.sha256

The baseline reads only the CT image and label. Probability maps contained in the archive are not passed to the baseline model.

## 1. Check the GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is not available. Enable a GPU in the Colab runtime settings."
    )

print(f"PyTorch version: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Mount Google Drive

The dataset archive is read from Google Drive. All experiment results are also saved there so they are not lost when the Colab runtime stops.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 3. Configure the experiment

This is the main configuration cell. Change the repository, archive location, experiment name, model, training, optimizer, or DataLoader settings here before running the next cell. `CT_PATH_COLUMN` intentionally uses `ct_parenchyma_path`.

In [ ]:
from pathlib import Path

# Repository
REPOSITORY_URL = (
    "https://github.com/FillipusAditya/"
    "mask-guided-lung-nodule-xai.git"
)
REPOSITORY_BRANCH = "refactor/002-segmentation"
PROJECT_ROOT = Path("/content/mask-guided-lung-nodule-xai")

# Experiment
EXPERIMENT_ID = "fa028196-fd4c-441f-a5d3-3efb9707e7f9"
EXPERIMENT_COMPONENT = "classification/cv_resnet50"

# Dataset
DRIVE_PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mask-guided-lung-nodule-xai"
)
ARCHIVE_NAME = f"{EXPERIMENT_ID}_guided_resnet50_parenchyma.tar.gz"
DRIVE_ARCHIVE_PATH = DRIVE_PROJECT_ROOT / ARCHIVE_NAME
DRIVE_CHECKSUM_PATH = DRIVE_PROJECT_ROOT / f"{ARCHIVE_NAME}.sha256"
LOCAL_ARCHIVE_PATH = Path("/content") / ARCHIVE_NAME
EXTRACTION_ROOT = (
    Path("/content") / f"cv_resnet50_training_data_{EXPERIMENT_ID}"
)
DATASET_ROOT = EXTRACTION_ROOT / "000_dataset_v2/_segmentation_dataset"
COPY_ARCHIVE_TO_LOCAL = True
VERIFY_ARCHIVE_SHA256 = True
FORCE_EXTRACT = False

# Output
DRIVE_OUTPUT_ROOT = DRIVE_PROJECT_ROOT / "experiment_results"
DRIVE_OUTPUT_DIR = (
    DRIVE_OUTPUT_ROOT / EXPERIMENT_ID / EXPERIMENT_COMPONENT
)

# Input data
METADATA_FILENAME = "004_classification_cv_5fold_seed42.csv"
CT_PATH_COLUMN = "ct_parenchyma_path"
INPUT_HEIGHT = 224
INPUT_WIDTH = 224
NUM_FOLDS = 5
CLASS_TO_IDX = {"benign": 0, "malignant": 1}
NORMALIZATION_MEAN = [0.485, 0.456, 0.406]
NORMALIZATION_STD = [0.229, 0.224, 0.225]

# Model ResNet-50
PRETRAINED_WEIGHTS = "DEFAULT"
CLASSIFIER_DROPOUT = 0.3

# Training
NUM_EPOCHS = 100
BATCH_SIZE = 32  # Reduce this value if the GPU runs out of memory.
LEARNING_RATE = 1e-3
SEED = 42
TRANSFORM_SEED = 42
CLASSIFICATION_THRESHOLD = 0.5
DEVICE = "auto"

# Optimizer SGD
MOMENTUM = 0.9
WEIGHT_DECAY = 1e-4
NESTEROV = False

# Optional plateau scheduler with an explicit LR sequence
SCHEDULER_ENABLED = False
SCHEDULER_LEARNING_RATES = [
    1e-3,
]
SCHEDULER_PATIENCE = 3
SCHEDULER_THRESHOLD = 1e-4
SCHEDULER_THRESHOLD_MODE = "rel"
SCHEDULER_COOLDOWN = 0

# Early stopping
EARLY_STOPPING_PATIENCE = 20
EARLY_STOPPING_MIN_DELTA = 0.0

# DataLoader
NUM_WORKERS = 2
PERSISTENT_WORKERS = True
PREFETCH_FACTOR = 2
PIN_MEMORY = True

# Optional testing after training
RUN_TEST_AFTER_TRAINING = True
MAX_TEST_SAMPLES = None  # Use 8 for a quick smoke test.
TEST_BATCH_SIZE = 1
TEST_NUM_WORKERS = 0

print(f"Archive: {DRIVE_ARCHIVE_PATH}")
print(f"Checksum: {DRIVE_CHECKSUM_PATH}")
print(f"Dataset: {DATASET_ROOT}")
print(f"Output: {DRIVE_OUTPUT_DIR}")

In [ ]:
# Runtime-only override: preserve the source archive and use a new output run.
SOURCE_EXPERIMENT_ID = EXPERIMENT_ID
RETRAINING_EXPERIMENT_ID = (
    f"{SOURCE_EXPERIMENT_ID}-retraining-reviewers-01"
)
EXPERIMENT_ID = RETRAINING_EXPERIMENT_ID
REPOSITORY_BRANCH = "main"
DRIVE_OUTPUT_DIR = (
    DRIVE_OUTPUT_ROOT / RETRAINING_EXPERIMENT_ID / EXPERIMENT_COMPONENT
)

print(f"Source experiment: {SOURCE_EXPERIMENT_ID}")
print(f"Retraining experiment: {RETRAINING_EXPERIMENT_ID}")
print(f"Repository branch: {REPOSITORY_BRANCH}")
print(f"Retraining output: {DRIVE_OUTPUT_DIR}")

## 4. Clone or update the repository

In [ ]:
import subprocess

if (PROJECT_ROOT / ".git").is_dir():
    print("Updating the existing repository...")
    subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), "fetch", "origin"],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), "checkout", REPOSITORY_BRANCH],
        check=True,
    )
    subprocess.run(
        [
            "git", "-C", str(PROJECT_ROOT), "pull", "--ff-only",
            "origin", REPOSITORY_BRANCH,
        ],
        check=True,
    )
else:
    print("Cloning the repository...")
    subprocess.run(
        [
            "git", "clone", "--branch", REPOSITORY_BRANCH,
            "--single-branch", REPOSITORY_URL, str(PROJECT_ROOT),
        ],
        check=True,
    )

training_script = PROJECT_ROOT / "003_classification/cv_resnet50/train.py"
if not training_script.is_file():
    raise FileNotFoundError(f"Training script not found: {training_script}")

print(f"Repository is ready: {PROJECT_ROOT}")

## 5. Install the required packages

The PyTorch installation provided by Colab is kept because it already matches the CUDA version in the active runtime.

In [ ]:
import sys

packages = [
    "albumentations>=2.0,<3.0",
    "opencv-python-headless",
    "pandas",
    "matplotlib",
    "scikit-learn",
    "tqdm",
    "zennit==0.5.1",
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", *packages],
    check=True,
)

print("All dependencies are ready.")

## 6. Copy, verify, and extract the archive

The archive is copied from Google Drive to Colab local storage. Its SHA-256 digest is checked against the accompanying checksum file before extraction. Copying, hashing, and extraction all display tqdm progress bars.

In [ ]:
import hashlib
import hmac
import shutil
import tarfile

from tqdm.auto import tqdm


def copy_file_with_progress(source, destination, chunk_size=8 * 1024 * 1024):
    """Copy one file while displaying byte-level progress."""

    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = destination.with_suffix(destination.suffix + ".part")
    total_bytes = source.stat().st_size

    with source.open("rb") as input_file:
        with temporary_path.open("wb") as output_file:
            with tqdm(
                total=total_bytes,
                desc="Copying archive",
                unit="B",
                unit_scale=True,
            ) as progress_bar:
                while True:
                    chunk = input_file.read(chunk_size)
                    if not chunk:
                        break
                    output_file.write(chunk)
                    progress_bar.update(len(chunk))

    temporary_path.replace(destination)


def sha256_with_progress(source, chunk_size=8 * 1024 * 1024):
    """Calculate SHA-256 while displaying byte-level progress."""

    digest = hashlib.sha256()
    total_bytes = source.stat().st_size
    with source.open("rb") as input_file:
        with tqdm(
            total=total_bytes,
            desc="Verifying SHA-256",
            unit="B",
            unit_scale=True,
        ) as progress_bar:
            while True:
                chunk = input_file.read(chunk_size)
                if not chunk:
                    break
                digest.update(chunk)
                progress_bar.update(len(chunk))
    return digest.hexdigest()


if not DRIVE_ARCHIVE_PATH.is_file():
    raise FileNotFoundError(f"Dataset archive not found: {DRIVE_ARCHIVE_PATH}")
if VERIFY_ARCHIVE_SHA256 and not DRIVE_CHECKSUM_PATH.is_file():
    raise FileNotFoundError(f"Checksum file not found: {DRIVE_CHECKSUM_PATH}")

if COPY_ARCHIVE_TO_LOCAL:
    archive_path = LOCAL_ARCHIVE_PATH
    source_size = DRIVE_ARCHIVE_PATH.stat().st_size
    local_copy_is_current = (
        archive_path.is_file() and archive_path.stat().st_size == source_size
    )
    if local_copy_is_current:
        print(f"Using local archive: {archive_path}")
    else:
        copy_file_with_progress(DRIVE_ARCHIVE_PATH, archive_path)
else:
    archive_path = DRIVE_ARCHIVE_PATH

archive_sha256 = None
if VERIFY_ARCHIVE_SHA256:
    expected_sha256 = (
        DRIVE_CHECKSUM_PATH.read_text(encoding="utf-8").split()[0].lower()
    )
    archive_sha256 = sha256_with_progress(archive_path)
    if not hmac.compare_digest(archive_sha256, expected_sha256):
        raise RuntimeError(
            f"SHA-256 mismatch: expected {expected_sha256}, "
            f"received {archive_sha256}"
        )
    print(f"SHA-256 valid: {archive_sha256}")

marker_id = archive_sha256[:16] if archive_sha256 else str(archive_path.stat().st_size)
extraction_marker = EXTRACTION_ROOT / f".extracted_{marker_id}"

if FORCE_EXTRACT and EXTRACTION_ROOT.exists():
    shutil.rmtree(EXTRACTION_ROOT)

if extraction_marker.is_file():
    print(f"Using local dataset: {EXTRACTION_ROOT}")
else:
    EXTRACTION_ROOT.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive_path, mode="r:gz") as archive:
        members = archive.getmembers()
        for member in tqdm(members, desc="Extracting archive", unit="file"):
            archive.extract(member, path=EXTRACTION_ROOT, filter="data")

    for old_marker in EXTRACTION_ROOT.glob(".extracted_*"):
        old_marker.unlink()
    extraction_marker.write_text(archive_path.name + "\n", encoding="utf-8")

if not DATASET_ROOT.is_dir():
    raise FileNotFoundError(f"Dataset root was not extracted: {DATASET_ROOT}")

print(f"Dataset is ready: {DATASET_ROOT}")

## 7. Validate the `000_dataset_v2` dataset

This cell checks the CV metadata columns, five-fold assignment, classes, and all parenchyma files used for training. Masks are also checked because they are required when the optional XAI test is run.

In [ ]:
import csv

metadata_path = DATASET_ROOT / METADATA_FILENAME
required_directories = [
    EXTRACTION_ROOT / "000_dataset_v2/_lidc/007_segmentation_dataset_npy/ct_parenchyma",
    EXTRACTION_ROOT / "000_dataset_v2/_lndb/007_segmentation_dataset_npy/ct_parenchyma",
    EXTRACTION_ROOT / "000_dataset_v2/_lidc/007_segmentation_dataset_npy/mask",
    EXTRACTION_ROOT / "000_dataset_v2/_lndb/007_segmentation_dataset_npy/mask",
]
required_columns = {
    "dataset", "patient_id", "filename", CT_PATH_COLUMN,
    "mask_path", "label", "cv_group_id", "cv_nodule_id",
    "cv_role", "cv_fold",
}

if not metadata_path.is_file():
    raise FileNotFoundError(f"Metadata not found: {metadata_path}")

for directory in required_directories:
    if not directory.is_dir():
        raise FileNotFoundError(f"Directory not found: {directory}")

with metadata_path.open("r", encoding="utf-8", newline="") as file:
    reader = csv.DictReader(file)
    column_names = set(reader.fieldnames or [])
    rows = list(reader)

missing_columns = required_columns - column_names
if missing_columns:
    raise ValueError(f"Metadata columns are incomplete: {sorted(missing_columns)}")
if not rows:
    raise ValueError("Metadata must not be empty.")

missing_files = []
for row in tqdm(rows, desc="Checking dataset paths", unit="sample"):
    ct_path = DATASET_ROOT / row[CT_PATH_COLUMN]
    mask_path = DATASET_ROOT / row["mask_path"]

    if not ct_path.is_file():
        missing_files.append(ct_path)
    if not mask_path.is_file():
        missing_files.append(mask_path)

if missing_files:
    examples = "\n".join(str(path) for path in missing_files[:5])
    raise FileNotFoundError(
        f"There are {len(missing_files)} missing files.\n{examples}"
    )

development_folds = {
    int(row["cv_fold"])
    for row in rows
    if row["cv_role"].strip().lower() == "development"
}
if development_folds != set(range(NUM_FOLDS)):
    raise ValueError(f"Unexpected folds: {sorted(development_folds)}")

label_counts = {}
for row in rows:
    label = row["label"].strip().lower()
    label_counts[label] = label_counts.get(label, 0) + 1

print(f"Number of samples: {len(rows):,}")
print(f"Class distribution: {label_counts}")
print(f"Development folds: {sorted(development_folds)}")
print("All parenchyma and mask files are available.")

## 8. Create and save the JSON configuration

The dictionary below is created directly from the main configuration cell. The JSON file is saved to the temporary repository so `train.py` can read it, and a persistent copy is saved to Google Drive. When training starts, `train.py` also copies the same JSON file to the experiment output directory as `cv_resnet50.json`.

In [ ]:
import json

if NUM_FOLDS != 5:
    raise ValueError("This metadata requires NUM_FOLDS = 5.")
if BATCH_SIZE < 1 or NUM_EPOCHS < 1 or LEARNING_RATE <= 0:
    raise ValueError("Batch size, epochs, and learning rate must be positive.")
if NUM_WORKERS < 0:
    raise ValueError("NUM_WORKERS must not be negative.")
if not 0.0 <= CLASSIFICATION_THRESHOLD <= 1.0:
    raise ValueError("CLASSIFICATION_THRESHOLD must be between 0 and 1.")
if DEVICE not in {"auto", "cpu", "cuda"}:
    raise ValueError("DEVICE must be 'auto', 'cpu', or 'cuda'.")
if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("DEVICE='cuda', but a GPU is not available.")
if not SCHEDULER_LEARNING_RATES:
    raise ValueError("SCHEDULER_LEARNING_RATES must not be empty.")
if any(value <= 0.0 for value in SCHEDULER_LEARNING_RATES):
    raise ValueError("Every scheduler learning rate must be positive.")
if any(
    current <= following
    for current, following in zip(
        SCHEDULER_LEARNING_RATES,
        SCHEDULER_LEARNING_RATES[1:],
    )
):
    raise ValueError("SCHEDULER_LEARNING_RATES must be strictly decreasing.")
if abs(SCHEDULER_LEARNING_RATES[0] - LEARNING_RATE) > 1e-15:
    raise ValueError("LEARNING_RATE must equal SCHEDULER_LEARNING_RATES[0].")
if SCHEDULER_PATIENCE < 0 or SCHEDULER_COOLDOWN < 0:
    raise ValueError("Scheduler patience and cooldown cannot be negative.")
if SCHEDULER_THRESHOLD < 0.0:
    raise ValueError("SCHEDULER_THRESHOLD cannot be negative.")
if SCHEDULER_THRESHOLD_MODE not in {"rel", "abs"}:
    raise ValueError("SCHEDULER_THRESHOLD_MODE must be 'rel' or 'abs'.")
config = {
    "experiment": {
        "id": EXPERIMENT_ID,
        "component": EXPERIMENT_COMPONENT,
    },
    "output": {
        "root_directory": str(DRIVE_OUTPUT_ROOT),
        "config_snapshot_filename": "cv_resnet50.json",
    },
    "data": {
        "dataset_root": str(DATASET_ROOT),
        "metadata_path": str(metadata_path),
        "ct_path_column": CT_PATH_COLUMN,
        "input_height": INPUT_HEIGHT,
        "input_width": INPUT_WIDTH,
        "class_to_idx": CLASS_TO_IDX,
        "normalization_mean": NORMALIZATION_MEAN,
        "normalization_std": NORMALIZATION_STD,
    },
    "cross_validation": {
        "num_folds": NUM_FOLDS,
        "development_role": "development",
        "holdout_role": "holdout_test",
        "holdout_fold": -1,
        "group_column": "cv_group_id",
        "nodule_column": "cv_nodule_id",
        "fold_column": "cv_fold",
        "role_column": "cv_role",
    },
    "model": {
        "architecture": "ResNet50",
        "pretrained_weights": PRETRAINED_WEIGHTS,
        "training_strategy": "full_fine_tuning",
        "trainable_component": "entire_model",
        "classifier_dropout": CLASSIFIER_DROPOUT,
    },
    "training": {
        "num_epochs": NUM_EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "seed": SEED,
        "transform_seed": TRANSFORM_SEED,
        "classification_threshold": CLASSIFICATION_THRESHOLD,
        "device": DEVICE,
    },
    "optimizer": {
        "name": "SGD",
        "momentum": MOMENTUM,
        "weight_decay": WEIGHT_DECAY,
        "nesterov": NESTEROV,
    },
    "scheduler": {
        "enabled": SCHEDULER_ENABLED,
        "name": "PlateauLearningRateSequence",
        "mode": "min",
        "learning_rates": SCHEDULER_LEARNING_RATES,
        "patience": SCHEDULER_PATIENCE,
        "threshold": SCHEDULER_THRESHOLD,
        "threshold_mode": SCHEDULER_THRESHOLD_MODE,
        "cooldown": SCHEDULER_COOLDOWN,
    },
    "dataloader": {
        "num_workers": NUM_WORKERS,
        "persistent_workers": PERSISTENT_WORKERS,
        "prefetch_factor": PREFETCH_FACTOR,
        "pin_memory": PIN_MEMORY,
        "train_shuffle": True,
        "val_shuffle": False,
        "train_drop_last": False,
        "val_drop_last": False,
    },
    "early_stopping": {
        "enabled": True,
        "monitor": "val_loss",
        "mode": "min",
        "patience": EARLY_STOPPING_PATIENCE,
        "min_delta": EARLY_STOPPING_MIN_DELTA,
        "verbose": True,
        "restore_best_weights": True,
    },
    "amp": {"training_enabled": False},
    "checkpoint": {"save_latest": True},
}


def save_json(data, output_path):
    """Save a dictionary as a readable JSON file."""

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as file:
        json.dump(data, file, indent=4)
        file.write("\n")


config_path = PROJECT_ROOT / "003_classification/configs/cv_resnet50.json"
drive_config_path = (
    DRIVE_OUTPUT_ROOT / "saved_configs" / EXPERIMENT_ID / "cv_resnet50.json"
)
save_json(config, config_path)
save_json(config, drive_config_path)

print(json.dumps(config, indent=4))
print(f"Config repository: {config_path}")
print(f"Config Google Drive: {drive_config_path}")

## 9. Run the preflight check

The preflight check uses the same configuration and transforms as training. This cell loads one validation sample, builds ResNet-50, and verifies the model input and output shapes. The pretrained ImageNet weights are downloaded when the model is first created.

In [ ]:
import importlib

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

importlib.invalidate_caches()
module_name = "003_classification.cv_resnet50.train"
if module_name in sys.modules:
    train_module = importlib.reload(sys.modules[module_name])
else:
    train_module = importlib.import_module(module_name)

train_loader, val_loader, _, _ = (
    train_module.build_fold_dataloaders(fold=0)
)
sample_image, sample_label = val_loader.dataset[0]
model = train_module.build_model(
    num_classes=len(CLASS_TO_IDX)
)
total_parameters = sum(parameter.numel() for parameter in model.parameters())
trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)
if trainable_parameters != total_parameters:
    raise RuntimeError(
        f"Full fine-tuning check failed: {trainable_parameters:,} of "
        f"{total_parameters:,} parameters are trainable."
    )
model.eval()

with torch.no_grad():
    sample_output = model(
        sample_image.unsqueeze(0).to(train_module.DEVICE)
    )

print(f"Fold 0 training samples: {len(train_loader.dataset):,}")
print(f"Fold 0 validation samples: {len(val_loader.dataset):,}")
print(f"Input shape: {tuple(sample_image.shape)}")
print(f"Output shape: {tuple(sample_output.shape)}")
print(f"Trainable parameters: {trainable_parameters:,}/{total_parameters:,}")
print(f"Sample label: {int(sample_label)}")

del model
del train_loader
del val_loader
torch.cuda.empty_cache()

## 10. Start full fine-tuning with 5-fold CV

Each fold starts from a new ImageNet-pretrained ResNet-50. Every backbone and classifier parameter is trainable. Training uses the same fold definitions and validation-loss model selection as the direct-guided comparison. tqdm shows training and validation progress.

In [ ]:
if DRIVE_OUTPUT_DIR.exists():
    raise FileExistsError(
        f"Output already exists: {DRIVE_OUTPUT_DIR}\n"
        "Change EXPERIMENT_ID to start a new training run."
    )

print("Starting baseline ResNet-50...", flush=True)
print(f"Input: {CT_PATH_COLUMN}", flush=True)
print(f"Fold: {NUM_FOLDS}", flush=True)
print(f"Epochs per fold: {NUM_EPOCHS}", flush=True)
print(f"Scheduler enabled: {SCHEDULER_ENABLED}", flush=True)
print(f"Output: {DRIVE_OUTPUT_DIR}", flush=True)

train_module.main()

## 11. View the cross-validation results

This cell displays the summary for each fold, the combined plot, and the locations of important artifacts.

In [ ]:
import pandas as pd
from IPython.display import Image, display

summary_path = DRIVE_OUTPUT_DIR / "cv_summary.csv"
plot_path = DRIVE_OUTPUT_DIR / "figures/cv_fold_metrics.png"
config_snapshot_path = DRIVE_OUTPUT_DIR / "cv_resnet50.json"
oof_predictions_path = DRIVE_OUTPUT_DIR / "out_of_fold_predictions.csv"

if not summary_path.is_file():
    raise FileNotFoundError(f"Training summary not found: {summary_path}")

display(pd.read_csv(summary_path))
if plot_path.is_file():
    display(Image(filename=str(plot_path), width=750))

print(f"Output directory: {DRIVE_OUTPUT_DIR}")
print(f"Config snapshot: {config_snapshot_path}")
print(f"OOF predictions: {oof_predictions_path}")
for fold in range(NUM_FOLDS):
    best_model_path = DRIVE_OUTPUT_DIR / f"fold_{fold}/best_model.pth"
    print(f"Fold {fold} best model: {best_model_path}")

## 12. Holdout ensemble and layer-wise XAI

With RUN_TEST_AFTER_TRAINING=True, the notebook evaluates the ensemble of five best fold checkpoints on the independent holdout set. It saves Grad-CAM and LRP maps for layer1, layer2, layer3, and layer4, then creates one ten-panel visualization per study. Set MAX_TEST_SAMPLES=8 only for a quick smoke test.

In [ ]:
if RUN_TEST_AFTER_TRAINING:
    test_command = [
        sys.executable,
        "-m",
        "003_classification.cv_resnet50.test",
        str(DRIVE_OUTPUT_DIR),
        "--batch-size",
        str(TEST_BATCH_SIZE),
        "--num-workers",
        str(TEST_NUM_WORKERS),
        "--device",
        DEVICE,
        "--dpi",
        "120",
    ]

    if MAX_TEST_SAMPLES is not None:
        test_command.extend(["--max-samples", str(MAX_TEST_SAMPLES)])

    subprocess.run(test_command, cwd=PROJECT_ROOT, check=True)
    print(f"Test results: {DRIVE_OUTPUT_DIR / 'test'}")
else:
    print("Testing was skipped because RUN_TEST_AFTER_TRAINING=False.")

## Run a new experiment

The current CV program creates a new output directory and does not automatically resume an interrupted fold. To run a new experiment, change `EXPERIMENT_ID`, adjust the configuration, and rerun the notebook starting from the configuration cell. Do not delete previous results from Google Drive until all required artifacts have been saved.

# Additional Reviewer Evaluations

The 27 original source cells are preserved exactly and in their original order. One runtime-only override cell is inserted immediately after the original configuration cell so the copied pipeline checks out `main` and writes to a separate retraining experiment directory. The source archive and saved U-Net probability maps continue to use the original experiment identifier. Reviewer-specific functionality is appended after the complete original pipeline.


## 13. Additional Evaluation Configuration

In [ ]:
REVIEWER_MODEL_NAME = "Baseline"
REVIEWER_COMPONENT = "classification/cv_resnet50"
COUNTERPART_COMPONENT = "classification/multistage_direct_guided_cv_resnet50"
IS_GUIDED_MODEL = False

REVIEWER_FIGURE_DPI = 500
CALIBRATION_BINS = 10
BOOTSTRAP_ITERATIONS = 10_000
BOOTSTRAP_SEED = 42

# Set this to True only when held-out predictions are missing and the
# five saved best-fold checkpoints should be used for inference.
RUN_REVIEWER_HELDOUT_INFERENCE = False
FORCE_REVIEWER_HELDOUT_INFERENCE = False

RETRAINING_EXPERIMENT_ROOT = (
    DRIVE_OUTPUT_ROOT / RETRAINING_EXPERIMENT_ID
)
REVIEWER_MODEL_DIR = RETRAINING_EXPERIMENT_ROOT / REVIEWER_COMPONENT
COUNTERPART_MODEL_DIR = (
    RETRAINING_EXPERIMENT_ROOT / COUNTERPART_COMPONENT
)
ORIGINAL_MODEL_DIR = (
    DRIVE_OUTPUT_ROOT / SOURCE_EXPERIMENT_ID / REVIEWER_COMPONENT
)
COMPARISON_OUTPUT_DIR = (
    RETRAINING_EXPERIMENT_ROOT / "evaluation/baseline_vs_proposed"
)

if REVIEWER_MODEL_DIR != DRIVE_OUTPUT_DIR:
    raise RuntimeError(
        "The retraining and reviewer output directories do not match."
    )
if REVIEWER_FIGURE_DPI != 500:
    raise ValueError("Reviewer figures must be exported at 500 DPI.")

print(f"Retraining experiment: {RETRAINING_EXPERIMENT_ID}")
print(f"Model output: {REVIEWER_MODEL_DIR}")
print(f"Counterpart output: {COUNTERPART_MODEL_DIR}")

## 14. Split Integrity and Saved-Artifact Validation

In [ ]:
import pandas as pd

reviewer_metadata = pd.read_csv(metadata_path)
reviewer_required_columns = {
    "dataset", "patient_id", "filename", "label", "cv_group_id",
    "cv_nodule_id", "cv_role", "cv_fold",
}
reviewer_missing_columns = (
    reviewer_required_columns - set(reviewer_metadata.columns)
)
if reviewer_missing_columns:
    raise ValueError(
        f"Missing metadata columns: {sorted(reviewer_missing_columns)}"
    )

reviewer_metadata = reviewer_metadata.copy()
reviewer_metadata["label"] = (
    reviewer_metadata["label"].astype(str).str.lower()
)
reviewer_metadata["unique_patient_id"] = (
    reviewer_metadata["dataset"].astype(str)
    + "::"
    + reviewer_metadata["patient_id"].astype(str)
)
if reviewer_metadata[["dataset", "filename"]].duplicated().any():
    raise ValueError("Duplicate slice identifiers were found.")
if not reviewer_metadata["label"].isin(["benign", "malignant"]).all():
    raise ValueError("Invalid classification labels were found.")

development_metadata = reviewer_metadata.loc[
    reviewer_metadata["cv_role"].eq("development")
]
held_out_metadata = reviewer_metadata.loc[
    reviewer_metadata["cv_role"].eq("holdout_test")
]
development_patients = set(development_metadata["unique_patient_id"])
held_out_patients = set(held_out_metadata["unique_patient_id"])
if development_patients & held_out_patients:
    raise RuntimeError("Development and held-out patients overlap.")
if not held_out_metadata["cv_fold"].astype(int).eq(-1).all():
    raise RuntimeError("Held-out samples have unexpected fold values.")

for validation_fold in range(NUM_FOLDS):
    training_patients = set(
        development_metadata.loc[
            development_metadata["cv_fold"].astype(int).ne(validation_fold),
            "unique_patient_id",
        ]
    )
    validation_patients = set(
        development_metadata.loc[
            development_metadata["cv_fold"].astype(int).eq(validation_fold),
            "unique_patient_id",
        ]
    )
    if training_patients & validation_patients:
        raise RuntimeError(
            f"Patient overlap was found in fold {validation_fold}."
        )

checkpoint_name = "best_model.pt" if IS_GUIDED_MODEL else "best_model.pth"
checkpoint_paths = [
    REVIEWER_MODEL_DIR / f"fold_{fold}" / checkpoint_name
    for fold in range(NUM_FOLDS)
]
missing_checkpoints = [
    path for path in checkpoint_paths if not path.is_file()
]
if missing_checkpoints:
    raise FileNotFoundError(
        f"Missing best-fold checkpoints: {missing_checkpoints}"
    )

print(f"Development patients: {len(development_patients):,}")
print(f"Held-out patients: {len(held_out_patients):,}")
print("Patient isolation and five best-fold checkpoints are valid.")

## 15. Evaluation-Only Held-Out Inference

The original 15% held-out cohort is evaluated by the five best fold
checkpoints. Existing prediction CSV files are reused by default, so
this section can be rerun without classifier retraining. The proposed
model reads the probability maps already stored in the source archive;
U-Net is not retrained.

In [ ]:
reviewer_test_predictions_path = (
    REVIEWER_MODEL_DIR / "test/test_predictions.csv"
)
should_run_inference = (
    RUN_REVIEWER_HELDOUT_INFERENCE
    or not reviewer_test_predictions_path.is_file()
)

if should_run_inference:
    if IS_GUIDED_MODEL:
        inference_command = [
            sys.executable,
            "-m",
            (
                "003_classification."
                "multistage_direct_guided_cv_resnet50.test"
            ),
            str(REVIEWER_MODEL_DIR),
            "--dataset-root", str(DATASET_ROOT),
            "--metadata-path", str(metadata_path),
            "--probability-root", str(PROBABILITY_ROOT),
            "--batch-size", str(TEST_BATCH_SIZE),
            "--num-workers", str(TEST_NUM_WORKERS),
            "--device", DEVICE,
            "--dpi", str(REVIEWER_FIGURE_DPI),
        ]
    else:
        inference_command = [
            sys.executable,
            "-m", "003_classification.cv_resnet50.test",
            str(REVIEWER_MODEL_DIR),
            "--dataset-root", str(DATASET_ROOT),
            "--metadata-path", str(metadata_path),
            "--batch-size", str(TEST_BATCH_SIZE),
            "--num-workers", str(TEST_NUM_WORKERS),
            "--device", DEVICE,
            "--dpi", str(REVIEWER_FIGURE_DPI),
        ]
        if FORCE_REVIEWER_HELDOUT_INFERENCE:
            inference_command.append("--force-inference")
    subprocess.run(inference_command, cwd=PROJECT_ROOT, check=True)

if not reviewer_test_predictions_path.is_file():
    raise FileNotFoundError(
        f"Held-out predictions not found: "
        f"{reviewer_test_predictions_path}"
    )
held_out_predictions = pd.read_csv(reviewer_test_predictions_path)
if not held_out_predictions["cv_role"].eq("holdout_test").all():
    raise RuntimeError("Non-held-out rows were found in test predictions.")
if held_out_predictions["cv_group_id"].isin(
    development_metadata["cv_group_id"]
).any():
    raise RuntimeError("A development patient entered held-out inference.")
prediction_keys = ["dataset", "patient_id", "cv_nodule_id", "filename"]
if held_out_predictions.duplicated(prediction_keys).any():
    raise RuntimeError("Duplicate held-out prediction rows were found.")

print(f"Held-out ensemble slices: {len(held_out_predictions):,}")
print(f"Predictions: {reviewer_test_predictions_path}")

## 16. Nodule-Level Majority Voting and Prediction Export

Primary nodule labels use the original majority vote across slices.
Exact ties are resolved by the mean malignant probability at threshold
0.5. The mean slice probability is retained only as the continuous
nodule score for ROC/PR and calibration analyses.

In [ ]:
import importlib

voting_module = importlib.import_module(
    "004_evaluation.classification_nodule_voting"
)
if IS_GUIDED_MODEL:
    validation_prediction_path = (
        REVIEWER_MODEL_DIR
        / "nodule_voting/validation_slice_predictions.csv"
    )
    if not validation_prediction_path.is_file():
        voting_module.infer_guided_validation(
            guided_dir=REVIEWER_MODEL_DIR,
            output_path=validation_prediction_path,
            dataset_root=DATASET_ROOT,
            metadata_path=metadata_path,
            probability_root=PROBABILITY_ROOT,
            requested_device=DEVICE,
            batch_size=BATCH_SIZE,
            num_workers=NUM_WORKERS,
        )
else:
    validation_prediction_path = (
        REVIEWER_MODEL_DIR / "out_of_fold_predictions.csv"
    )

voting_module.write_model_outputs(
    model_dir=REVIEWER_MODEL_DIR,
    validation_source=validation_prediction_path,
    test_source=reviewer_test_predictions_path,
)
reviewer_nodule_predictions_path = (
    REVIEWER_MODEL_DIR
    / "nodule_voting/test_nodule_predictions.csv"
)
print(f"Nodule predictions: {reviewer_nodule_predictions_path}")

## 17. Source-Specific, Confusion-Matrix, ROC/PR, and Calibration Evaluation

Combined, LIDC-IDRI-only, and LNDb-only results are subsets of the
same held-out ensemble predictions. Calibration uses ten equal-width
bins on [0, 1]. ECE is the sample-weighted mean absolute difference
between each non-empty bin's mean predicted probability and observed
malignant fraction. No calibration model or probability adjustment is
fitted.

In [ ]:
evaluation_module = importlib.import_module(
    "004_evaluation.retraining_classification_evaluation"
)
reviewer_outputs = evaluation_module.run_single_model_evaluation(
    model_dir=REVIEWER_MODEL_DIR,
    model_name=REVIEWER_MODEL_NAME,
    metadata_path=metadata_path,
    calibration_bins=CALIBRATION_BINS,
    dpi=REVIEWER_FIGURE_DPI,
)
for output_name, output_path in reviewer_outputs.items():
    print(f"{output_name}: {output_path}")

## 18. Reproduction Difference Report

In [ ]:
original_prediction_paths = {
    "slice": ORIGINAL_MODEL_DIR / "test/test_predictions.csv",
    "nodule": (
        ORIGINAL_MODEL_DIR
        / "nodule_voting/test_nodule_predictions.csv"
    ),
}
if all(path.is_file() for path in original_prediction_paths.values()):
    current_metrics = pd.read_csv(
        REVIEWER_MODEL_DIR
        / "source_specific_evaluation/source_specific_metrics.csv"
    )
    reproduction_frames = []
    for evaluation_level, prediction_path in (
        original_prediction_paths.items()
    ):
        original_predictions = evaluation_module.prepare_predictions(
            prediction_path, evaluation_level
        )
        original_metrics = evaluation_module.source_metric_table(
            original_predictions,
            REVIEWER_MODEL_NAME,
            evaluation_level,
        )
        current_level_metrics = current_metrics.loc[
            current_metrics["evaluation_level"].eq(evaluation_level)
        ]
        comparison = original_metrics.merge(
            current_level_metrics,
            on=["model", "dataset_scope", "evaluation_level"],
            suffixes=("_original", "_retraining"),
            validate="one_to_one",
        )
        for metric_name in evaluation_module.METRIC_NAMES:
            comparison[f"difference_{metric_name}"] = (
                comparison[f"{metric_name}_retraining"]
                - comparison[f"{metric_name}_original"]
            )
        reproduction_frames.append(comparison)

    reproduction_report = pd.concat(
        reproduction_frames, ignore_index=True
    )
    reproduction_path = (
        REVIEWER_MODEL_DIR
        / "source_specific_evaluation/"
        "reproduction_difference_report.csv"
    )
    reproduction_report.to_csv(reproduction_path, index=False)
    display(reproduction_report)
    print(f"Reproduction report: {reproduction_path}")
else:
    print(
        "Reproduction comparison was skipped because original "
        "slice- or nodule-level predictions were unavailable."
    )

## 19. Paired Patient-Clustered Bootstrap Comparison

Patients are sampled with replacement for 10,000 iterations using
seed 42. Every selected patient's nodules are included, and the same
sampled patients are used for both models. Reported intervals are the
2.5th and 97.5th percentiles of Proposed minus Baseline differences.
Undefined class-specific metrics remain missing rather than being set
to zero.

In [ ]:
counterpart_required_outputs = [
    COUNTERPART_MODEL_DIR / "test/test_predictions.csv",
    (
        COUNTERPART_MODEL_DIR
        / "nodule_voting/test_nodule_predictions.csv"
    ),
]
current_required_outputs = [
    reviewer_test_predictions_path,
    reviewer_nodule_predictions_path,
]
if all(
    path.is_file()
    for path in current_required_outputs + counterpart_required_outputs
):
    baseline_model_dir = (
        REVIEWER_MODEL_DIR
        if REVIEWER_MODEL_NAME == "Baseline"
        else COUNTERPART_MODEL_DIR
    )
    proposed_model_dir = (
        REVIEWER_MODEL_DIR
        if REVIEWER_MODEL_NAME == "Proposed"
        else COUNTERPART_MODEL_DIR
    )
    paired_outputs = evaluation_module.run_paired_comparison(
        baseline_dir=baseline_model_dir,
        proposed_dir=proposed_model_dir,
        output_dir=COMPARISON_OUTPUT_DIR,
        bootstrap_iterations=BOOTSTRAP_ITERATIONS,
        seed=BOOTSTRAP_SEED,
        dpi=REVIEWER_FIGURE_DPI,
    )
    for output_name, output_path in paired_outputs.items():
        print(f"{output_name}: {output_path}")
else:
    print(
        "Paired comparison is deferred until both retraining "
        "notebooks have exported slice- and nodule-level predictions."
    )

## 20. Results Summary and Export Validation

In [ ]:
from IPython.display import Image, display
from PIL import Image as PILImage

source_directory = REVIEWER_MODEL_DIR / "source_specific_evaluation"
required_reviewer_outputs = [
    source_directory / "source_class_split_distribution.csv",
    source_directory / "held_out_slice_predictions.csv",
    source_directory / "held_out_nodule_predictions.csv",
    source_directory / "source_specific_metrics.csv",
    source_directory / "source_specific_performance.png",
    (
        REVIEWER_MODEL_DIR
        / "confusion_matrices/classification_metrics_summary.csv"
    ),
    (
        REVIEWER_MODEL_DIR
        / "roc_pr_evaluation/roc_auc_average_precision_summary.csv"
    ),
    REVIEWER_MODEL_DIR / "calibration/calibration_summary.csv",
]
missing_reviewer_outputs = [
    path for path in required_reviewer_outputs if not path.is_file()
]
if missing_reviewer_outputs:
    raise FileNotFoundError(
        f"Missing reviewer outputs: {missing_reviewer_outputs}"
    )

exported_slice_predictions = pd.read_csv(
    source_directory / "held_out_slice_predictions.csv"
)
required_export_columns = {
    "dataset_source", "patient_id", "nodule_id", "slice_id",
    "ground_truth_label", "ensemble_malignant_probability",
    "predicted_label",
}
if required_export_columns - set(exported_slice_predictions.columns):
    raise RuntimeError("The slice prediction export is incomplete.")

additional_figure_directories = [
    source_directory,
    REVIEWER_MODEL_DIR / "confusion_matrices",
    REVIEWER_MODEL_DIR / "roc_pr_evaluation",
    REVIEWER_MODEL_DIR / "calibration",
    REVIEWER_MODEL_DIR / "training_curves",
]
additional_figures = [
    path
    for directory in additional_figure_directories
    for path in directory.glob("*.png")
]
low_resolution_figures = []
for figure_path in additional_figures:
    with PILImage.open(figure_path) as image:
        dpi = image.info.get("dpi")
    if dpi is None or min(dpi) < 490:
        low_resolution_figures.append((figure_path, dpi))
if low_resolution_figures:
    raise RuntimeError(
        "Reviewer figures below the 500-DPI tolerance: "
        f"{low_resolution_figures[:5]}"
    )

display(pd.read_csv(source_directory / "source_specific_metrics.csv"))
display(
    pd.read_csv(
        REVIEWER_MODEL_DIR
        / "roc_pr_evaluation/roc_auc_average_precision_summary.csv"
    )
)
display(
    pd.read_csv(
        REVIEWER_MODEL_DIR / "calibration/calibration_summary.csv"
    )
)
display(
    Image(
        filename=str(
            source_directory / "source_specific_performance.png"
        ),
        width=900,
    )
)
print(f"Validated reviewer figures: {len(additional_figures):,}")
print(f"Reviewer output root: {REVIEWER_MODEL_DIR}")

## 21. Evaluation-Only Rerun

To rerun evaluation without training, execute the environment,
repository, dependency, archive, and dataset-validation cells above;
skip the original training cell; set
`RUN_REVIEWER_HELDOUT_INFERENCE = False`; then execute Sections 13-20.
Saved best-fold checkpoints and prediction CSV files are reused. Set
`RUN_REVIEWER_HELDOUT_INFERENCE = True` only when the held-out
prediction CSV must be reconstructed from the saved checkpoints.